# Qwen3.5-4B QLoRA — Kaggle training

This notebook is the Kaggle counterpart of `train_qwen_qlora.ipynb`.

It does **not** clone GitHub. Attach a Kaggle Dataset containing the current `LLM-APA-main` repository snapshot, and optionally attach a separate private Dataset containing lecturer data.

Kaggle's `/kaggle/input` is strictly read-only, so the repo snapshot is copied once to a writable runtime directory before installation or training:

```text
/kaggle/input/<repo-dataset>/LLM-APA-main/   # immutable attached snapshot
                    ↓ copy once
/kaggle/working/LLM-APA-main/                # writable runtime source
```

Recommended layout:

```text
/kaggle/input/<repo-dataset>/LLM-APA-main/     # attached repo snapshot
/kaggle/input/<private-data-dataset>/...        # optional real data
/kaggle/working/LLM-APA-main/                   # writable source copy
/kaggle/working/llm-apa-runtime/                # temporary venv + HF cache
/kaggle/working/LLM-APA-runs/                   # checkpoints / metrics
```

Flow:

```text
discover repo input → copy to working → environment/GPU preflight
→ prepare split → 2-step smoke gate → train/resume
→ validation predict/evaluate → optional lecturer-test package
```

`/kaggle/working` must be saved as Notebook Output if checkpoints need to survive the session.


## Before running

1. Select a Kaggle GPU accelerator.
2. Enable Internet if dependencies / Hugging Face weights are not already attached.
3. Attach the current repo snapshot as a Kaggle Dataset.
4. Attach lecturer data only if course policy allows uploading it to Kaggle; keep it private.
5. For resume, attach the previous Notebook Output as an input.

If auto-discovery finds more than one repo snapshot, set `REPO_DATASET_SLUG` to the Dataset slug (recommended), or set `REPO_INPUT_OVERRIDE` to a confirmed runtime path. Dataset discovery supports both flat and owner-qualified Kaggle mounts.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
if not INPUT.exists():
    raise RuntimeError("This notebook is intended for Kaggle.")

# Optional explicit path to the attached immutable repo snapshot.
REPO_INPUT_OVERRIDE = None
# Select one attached Dataset without depending on its runtime mount prefix.
REPO_DATASET_SLUG = None
# Writable runtime location used by uv/Python/training.
REPO_WORK = WORKING / "LLM-APA-main"

def find_attached_files(filename, dataset_slug=None):
    return [
        path for path in INPUT.glob(f"**/{filename}")
        if dataset_slug is None or dataset_slug in path.relative_to(INPUT).parts
    ]

def find_repo_inputs():
    found = []
    for pyproject in find_attached_files("pyproject.toml", REPO_DATASET_SLUG):
        root = pyproject.parent
        if (root / "scripts" / "train.py").exists() and (root / "src" / "llm_grading").exists():
            found.append(root)
    return list(dict.fromkeys(map(str, found)))

if REPO_INPUT_OVERRIDE:
    REPO_INPUT = Path(REPO_INPUT_OVERRIDE)
else:
    repos = find_repo_inputs()
    if not repos:
        raise FileNotFoundError(
            "No LLM-APA repo snapshot found under /kaggle/input. "
            "Attach a Kaggle Dataset containing LLM-APA-main."
        )
    if len(repos) > 1:
        raise RuntimeError(
            "Multiple repo snapshots found; set REPO_DATASET_SLUG or REPO_INPUT_OVERRIDE:\n"
            + "\n".join(repos)
        )
    REPO_INPUT = Path(repos[0])

required = [
    "pyproject.toml", "uv.lock", "scripts/prepare_data.py", "scripts/smoke.py",
    "scripts/train.py", "scripts/predict.py", "scripts/evaluate.py",
    "scripts/validate_predictions.py", "scripts/package_predictions.py",
]
missing = [p for p in required if not (REPO_INPUT / p).exists()]
if missing:
    raise FileNotFoundError(f"Repo input snapshot is incomplete: {missing}")

# Always refresh the writable copy at notebook start so code matches the attached snapshot.
if REPO_WORK.exists():
    shutil.rmtree(REPO_WORK)

shutil.copytree(
    REPO_INPUT,
    REPO_WORK,
    ignore=shutil.ignore_patterns(
        ".git", "__pycache__", "*.pyc", ".venv", "outputs"
    ),
)

REPO = REPO_WORK
missing_after_copy = [p for p in required if not (REPO / p).exists()]
if missing_after_copy:
    raise FileNotFoundError(f"Writable repo copy is incomplete: {missing_after_copy}")

print("Immutable repo input :", REPO_INPUT)
print("Writable repo copy   :", REPO)
print("Input is read-only    :", str(REPO_INPUT).startswith("/kaggle/input/"))
print("Training uses working :", str(REPO).startswith("/kaggle/working/"))


## Completion-only memory optimization and source checks

Qwen3.5 has a large vocabulary. The shared `CompletionCollator` sends `logits_to_keep` and aligned `shift_labels`, so training computes vocabulary logits only where they predict supervised answer tokens. Prompt evidence stays intact.

Attach an updated repository snapshot containing both `src/llm_grading/training/dataset.py` and `src/llm_grading/training/trainer.py`. The environment cell verifies the collator and smoke policy before loading Qwen. An older attached snapshot stops with an actionable error; update the Kaggle Dataset and restart from the copy cell.


In [ ]:
for relative in ("src/llm_grading/training/dataset.py", "src/llm_grading/training/trainer.py"):
    if not (REPO / relative).is_file():
        raise FileNotFoundError(f"Repo snapshot is missing {relative}")
print("Training uses shared source from:", REPO)


## Runtime environment

The attached repo is immutable under `/kaggle/input`, so it is copied once to `/kaggle/working/LLM-APA-main`.

`uv sync` and Python then operate only on the writable copy. This avoids build/install failures caused by setuptools/uv trying to create package metadata under `/kaggle/input`.

The virtual environment and Hugging Face cache also stay under `/kaggle/working`.


In [ ]:
RUNTIME_ROOT = WORKING/"llm-apa-runtime"
VENV = RUNTIME_ROOT/"venv"
UV_CACHE = RUNTIME_ROOT/"uv-cache"
HF_CACHE = RUNTIME_ROOT/"hf-cache"
for p in (RUNTIME_ROOT, UV_CACHE, HF_CACHE):
    p.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable,"-m","pip","install","-q","uv"], check=True)

SETUP_ENV = dict(os.environ,
    UV_PROJECT_ENVIRONMENT=str(VENV),
    UV_CACHE_DIR=str(UV_CACHE),
    PYTHONDONTWRITEBYTECODE="1",
)
if not str(REPO).startswith("/kaggle/working/"):
    raise RuntimeError(f"Refusing to install/build the project from a read-only path: {REPO}")

subprocess.run([sys.executable,"-m","uv","python","install","3.11"], env=SETUP_ENV, check=True)
subprocess.run([
    sys.executable,"-m","uv","sync","--frozen","--python","3.11",
    "--group","dev","--group","ml"
], cwd=REPO, env=SETUP_ENV, check=True)

PYTHON = VENV/"bin/python"
RUN_ENV = dict(os.environ,
    # Keep Trainer and completion-logit positions on a single GPU.
    CUDA_VISIBLE_DEVICES="0",
    UV_PROJECT_ENVIRONMENT=str(VENV),
    UV_CACHE_DIR=str(UV_CACHE),
    HF_HOME=str(HF_CACHE),
    HUGGINGFACE_HUB_CACHE=str(HF_CACHE/"hub"),
    TOKENIZERS_PARALLELISM="false",
    PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
    PYTHONDONTWRITEBYTECODE="1",
)

def run(script,*args):
    cmd=[str(PYTHON), str(REPO/"scripts"/script), *map(str,args)]
    print("+"," ".join(cmd))
    subprocess.run(cmd,cwd=REPO,env=RUN_ENV,check=True)

def py_eval(src):
    return subprocess.check_output([str(PYTHON),"-c",src],cwd=REPO,env=RUN_ENV,text=True).strip()

# Verify the attached source before spending time loading the full Qwen model.
source_checks = json.loads(py_eval("""
import json
from types import SimpleNamespace
import llm_grading.training.trainer as trainer_module
from llm_grading.training.dataset import CompletionCollator
batch = CompletionCollator(SimpleNamespace(pad_token_id=0))([
    {"input_ids": [1, 2, 3], "attention_mask": [1, 1, 1], "labels": [-100, -100, 3]}
])
print(json.dumps({
    "completion_only_logits": (
        "logits_to_keep" in batch and "shift_labels" in batch
        and batch["logits_to_keep"].tolist() == [1]
        and batch["shift_labels"].tolist() == [[3]]
    ),
    "smoke_overrides": getattr(trainer_module, "SMOKE_TRAINING_OVERRIDES", {}),
}))
"""))
expected_smoke_overrides = {
    "warmup_steps": 0,
    "load_best_model_at_end": False,
    "metric_for_best_model": None,
}
if (not source_checks["completion_only_logits"]
        or source_checks["smoke_overrides"] != expected_smoke_overrides):
    raise RuntimeError(
        "Attached repo snapshot is missing the completion-only memory or smoke-training fixes. "
        "Update the Kaggle repo Dataset with the corrected dataset.py and trainer.py, "
        "then rerun from the copy cell. Updating only this notebook is insufficient."
    )
print("Source checks passed:", source_checks)


## GPU preflight

The current QLoRA path requires CUDA. The notebook sets BF16 only when supported; otherwise it uses FP16.

The notebook sets `CUDA_VISIBLE_DEVICES=0` for every repository subprocess before it imports PyTorch. GPU preflight and training therefore see exactly one GPU, including on a multi-GPU accelerator.


In [ ]:
gpu_info=json.loads(py_eval(
"import json,torch; print(json.dumps({"
"'torch':torch.__version__,"
"'cuda':torch.cuda.is_available(),"
"'cuda_version':torch.version.cuda,"
"'gpu_count':torch.cuda.device_count(),"
"'gpu0':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"
"'vram0_gb':round(torch.cuda.get_device_properties(0).total_memory/2**30,2) if torch.cuda.is_available() else 0,"
"'bf16':bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())"
"}))"
))
print(json.dumps(gpu_info,indent=2))
if not gpu_info["cuda"]:
    raise RuntimeError("Enable a Kaggle GPU accelerator.")
if gpu_info["gpu_count"] != 1:
    raise RuntimeError("Expected one visible GPU; rerun the runtime environment cell.")
print("Repository subprocesses use physical GPU 0 only.")
print("dtype:", "bfloat16" if gpu_info["bf16"] else "float16")


## Experiment and data settings

Set `DATA_ROOT_OVERRIDE` when real lecturer data is attached as a separate private Kaggle Dataset.

Otherwise the notebook uses `data/raw` from the repo snapshot if populated, then checks `sample_dataset` (the documented sample layout) and `data/sample_dataset`. A source-only snapshot contains no lecturer/sample data; attach it separately and set `DATA_ROOT_OVERRIDE`.

For official train/dev paths, set both `TRAIN_FILE` and `VAL_FILE`.


In [ ]:
import yaml

TASK="task1"
VERSION="F0-v1"

DATA_ROOT_OVERRIDE=None
TRAIN_FILE=None
VAL_FILE=None
TEST_FILE=None

# Exact resume from attached previous Notebook Output
PREVIOUS_RUN_DIR=None
RESUME_LATEST=False
RESUME_FROM=None

repo_raw=REPO/"data/raw"
repo_samples=[REPO/"sample_dataset", REPO/"data/sample_dataset"]

if (TRAIN_FILE is None)!=(VAL_FILE is None):
    raise ValueError("Set TRAIN_FILE and VAL_FILE together.")

def contains_json(p):
    p=Path(p)
    return p.exists() and any(x.name!=".gitkeep" for x in p.rglob("*.json"))

repo_sample=next((path for path in repo_samples if contains_json(path)), None)

if DATA_ROOT_OVERRIDE:
    DATA_ROOT=Path(DATA_ROOT_OVERRIDE)
    USING_SAMPLE=False
elif TRAIN_FILE and VAL_FILE:
    # Official split files supply their own source; no repo dataset is required.
    DATA_ROOT=Path(TRAIN_FILE).parent
    USING_SAMPLE=False
elif contains_json(repo_raw):
    DATA_ROOT=repo_raw
    USING_SAMPLE=False
elif repo_sample is not None:
    DATA_ROOT=repo_sample
    USING_SAMPLE=True
else:
    raise FileNotFoundError("No dataset found. Attach one and set DATA_ROOT_OVERRIDE.")

DATASET_VERSION="sample-v1" if USING_SAMPLE else "lecturer-v1"
SPLIT_VERSION="grouped-v1"

RUNS_ROOT=WORKING/"LLM-APA-runs"
RUN_DIR=RUNS_ROOT/TASK/VERSION
PREPARED_DIR=RUNS_ROOT/"prepared"/TASK/f"{DATASET_VERSION}-{SPLIT_VERSION}"
RUN_DIR.mkdir(parents=True,exist_ok=True)
PREPARED_DIR.mkdir(parents=True,exist_ok=True)

print("Task:",TASK)
print("Version:",VERSION)
print("Data:",DATA_ROOT)
print("Mode:","SAMPLE PILOT" if USING_SAMPLE else "LECTURER/ATTACHED DATA")
print("Run dir:",RUN_DIR)


## Resume helper

For exact resume, attach the previous Notebook Output, set `PREVIOUS_RUN_DIR`, and enable `RESUME_LATEST`.

The checkpoint must contain Trainer state, optimizer and scheduler state. A final adapter-only folder is not an exact resume point.


In [ ]:
def latest_checkpoint(directory):
    if not directory:
        return None
    items=[]
    for p in Path(directory).glob("checkpoint-*"):
        try:
            step=int(p.name.rsplit("-",1)[1])
        except ValueError:
            continue
        need=["trainer_state.json","adapter_config.json","optimizer.pt","scheduler.pt"]
        if all((p/n).is_file() for n in need):
            items.append((step,p))
    return max(items)[1] if items else None

resume_path=Path(RESUME_FROM) if RESUME_FROM else None
if RESUME_LATEST:
    if not PREVIOUS_RUN_DIR:
        raise ValueError("Set PREVIOUS_RUN_DIR.")
    resume_path=latest_checkpoint(PREVIOUS_RUN_DIR)
    if resume_path is None:
        raise FileNotFoundError("No resumable checkpoint found.")

resume_revision=None
if resume_path:
    contract=resume_path/"training_contract.json"
    if not contract.exists():
        raise FileNotFoundError(contract)
    resume_revision=json.loads(contract.read_text())["model_revision"]
    print("Resume:",resume_path)
    print("Frozen base revision:",resume_revision)


## Build versioned run config

The immutable Kaggle Input snapshot is copied to a writable runtime repo. All config generation, package installation, prepared splits, checkpoints, manifests, predictions and metrics operate under `/kaggle/working`.

A changed hypothesis should use a new `VERSION`. Exact resume is only for the same training contract.


In [ ]:
base_config_path=REPO/f"configs/{TASK}/qwen_lora.yaml"
config=yaml.safe_load(base_config_path.read_text(encoding="utf-8"))

eid=f"{TASK}-{VERSION}"
config["experiment"].update(
    id=eid,
    name=eid.replace("-","_"),
    notes=("Kaggle sample pilot" if USING_SAMPLE else "Kaggle QLoRA baseline"),
)
config["seed"]=int(config.get("seed",42))
config["model"]["dtype"]="bfloat16" if gpu_info["bf16"] else "float16"
if resume_revision:
    config["model"]["revision"]=resume_revision

config["data"].update(
    input_path=str(DATA_ROOT),
    train_path=str(TRAIN_FILE) if TRAIN_FILE else None,
    validation_path=str(VAL_FILE) if VAL_FILE else None,
    test_path=str(TEST_FILE) if TEST_FILE else None,
    prepared_dir=str(PREPARED_DIR),
    dataset_version=DATASET_VERSION,
    split_version=SPLIT_VERSION,
)
config["saving"]["output_dir"]=str(RUN_DIR)
config["training"]["resume_from_checkpoint"]=None

REQUEST_CONFIG=RUN_DIR/"requested_config.json"
REQUEST_CONFIG.write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding="utf-8")
print("Config:",REQUEST_CONFIG)
print("max_sequence_length:",config["training"]["max_sequence_length"])


## Prepare and audit

The repository keeps exact normalized-code groups together in the local split and persists the split manifest.


In [ ]:
run("prepare_data.py","--config",REQUEST_CONFIG,"--training")

safe_id=re.sub(r"[^A-Za-z0-9_.-]","_",config["experiment"]["id"])
PREPARED=PREPARED_DIR/f"config-{safe_id}.json"
if not PREPARED.exists():
    raise FileNotFoundError(PREPARED)

audit=json.loads((PREPARED_DIR/"audit.json").read_text(encoding="utf-8"))
print("Prepared:",PREPARED)
print("Split counts:",audit.get("counts"))


## 2-step QLoRA smoke gate

Run this before full training. It checks model/tokenizer load, finite loss, actual LoRA updates, checkpoint save/reload, inference, schema and evaluator.

Smoke uses **zero warmup** and **keeps the final step's weights**. With two steps, a warmup step can have zero learning rate; tied validation scores can then select that unchanged checkpoint. Full training retains its configured warmup and checkpoint selection.

Each attempt gets a separate output folder, so rerunning this cell after a failure preserves earlier diagnostics and avoids stale checkpoint conflicts. If smoke fails, inspect the printed manifest and step logs. FP16 overflow can skip optimizer updates even with a finite loss; the LoRA check must still pass before full training. Completion-only logits reduce vocabulary memory without truncating evidence. The full-training gate binds a passed smoke to the current task, version and prepared config; rerun smoke after changing any of these.


In [ ]:
import copy
import hashlib
from uuid import uuid4

RUN_SMOKE = True
SMOKE_IDENTITY = None
if RUN_SMOKE:
    prepared_bytes = PREPARED.read_bytes()
    smoke_identity = (TASK, VERSION, hashlib.sha256(prepared_bytes).hexdigest())
    SMOKE_ATTEMPT_DIR = RUNS_ROOT / "smoke-checks" / TASK / VERSION / uuid4().hex[:12]
    SMOKE_ATTEMPT_DIR.mkdir(parents=True, exist_ok=False)
    smoke_config = copy.deepcopy(json.loads(prepared_bytes))
    smoke_config["saving"]["output_dir"] = str(SMOKE_ATTEMPT_DIR)
    # Smoke always starts fresh, independently of full-training resume/warm-start.
    smoke_config["training"]["resume_from_checkpoint"] = None
    smoke_config["training"]["warm_start_adapter"] = None
    SMOKE_CONFIG = SMOKE_ATTEMPT_DIR / "requested_config.json"
    SMOKE_CONFIG.write_text(json.dumps(smoke_config, indent=2, ensure_ascii=False), encoding="utf-8")
    SMOKE_DIR = SMOKE_ATTEMPT_DIR / "smoke"
    print("Smoke artifacts:", SMOKE_DIR)
    try:
        run("smoke.py", "--config", SMOKE_CONFIG, "--training")
    except subprocess.CalledProcessError:
        for filename in ("smoke_status.json", "run_manifest.json"):
            diagnostic = SMOKE_DIR / filename
            if diagnostic.exists():
                print(filename, diagnostic.read_text(encoding="utf-8"))
        for checkpoint in sorted(SMOKE_DIR.glob("checkpoint-*")):
            state_path = checkpoint / "trainer_state.json"
            if state_path.exists():
                state = json.loads(state_path.read_text(encoding="utf-8"))
                print(checkpoint.name, json.dumps(state.get("log_history", []), indent=2))
        raise
    status = json.loads((SMOKE_DIR / "smoke_status.json").read_text(encoding="utf-8"))
    print(json.dumps(status, indent=2))
    if status.get("status") != "passed":
        raise RuntimeError("Smoke failed; do not start full training.")
    SMOKE_IDENTITY = smoke_identity


## Train or resume

- Fresh version: no resume path.
- Interrupted same experiment: resume exact `checkpoint-*`.
- Changed data/prompt/LoRA settings: use a new `VERSION`.

The trainer independently validates the saved training contract before resuming.


In [ ]:
if globals().get("RUN_SMOKE", True):
    import hashlib

    current_identity = (TASK, VERSION, hashlib.sha256(PREPARED.read_bytes()).hexdigest())
    smoke_directory = globals().get("SMOKE_DIR")
    smoke_status_file = smoke_directory / "smoke_status.json" if smoke_directory else None
    if (globals().get("SMOKE_IDENTITY") != current_identity
            or smoke_status_file is None or not smoke_status_file.is_file()
            or json.loads(smoke_status_file.read_text(encoding="utf-8")).get("status") != "passed"):
        raise RuntimeError("Run the smoke cell successfully for the current task/version/config first; full training requires a passed smoke.")

args=["--config",PREPARED]
if resume_path:
    args += ["--resume-from-checkpoint",resume_path]

run("train.py",*args)

MANIFEST=RUN_DIR/"run_manifest.json"
manifest=json.loads(MANIFEST.read_text())
print("Status:",manifest.get("status"))
print("Model revision:",manifest.get("model_revision"))
print("Adapter:",manifest.get("adapter_path"))
print("Best checkpoint:",manifest.get("best_checkpoint"))
print("Peak VRAM:",manifest.get("peak_vram_bytes"))


## Freeze inference config and evaluate validation

The inference config uses the exact model commit and saved adapter. Evaluation runs only on reference-bearing validation data.


In [ ]:
if manifest.get("status")!="completed":
    raise RuntimeError("Training did not complete.")

inference=json.loads(PREPARED.read_text())
inference["model"].update(
    revision=manifest["model_revision"],
    adapter_path=manifest["adapter_path"],
)
inference["saving"]["output_dir"]=str(RUN_DIR)

INFERENCE=RUN_DIR/"inference_config.json"
INFERENCE.write_text(json.dumps(inference,indent=2,ensure_ascii=False))

VAL_PRED=RUN_DIR/"validation_predictions.json"
VAL_METRICS=RUN_DIR/"validation_metrics.json"

run("predict.py","--config",INFERENCE,"--split","val","--output",VAL_PRED)
run("validate_predictions.py","--task",TASK,"--input",VAL_PRED)
run("evaluate.py","--config",INFERENCE,"--split","val","--predictions",VAL_PRED,"--output",VAL_METRICS)

metrics=json.loads(VAL_METRICS.read_text())
print(json.dumps(metrics,indent=2,ensure_ascii=False))


## Lecturer/private test packaging

Set `TEST_FILE` only when lecturer test input is available. This stage performs inference → validation → challenge-format packaging; it does not evaluate or tune against private targets.


In [ ]:
if TEST_FILE:
    FINAL_PRED=RUN_DIR/"final_internal_predictions.json"
    SUBMISSION=RUN_DIR/"submission.zip"
    run("predict.py","--config",INFERENCE,"--input",TEST_FILE,"--output",FINAL_PRED)
    run("validate_predictions.py","--task",TASK,"--input",FINAL_PRED)
    run("package_predictions.py","--task",TASK,"--input",FINAL_PRED,"--output",SUBMISSION,"--format","challenge")
    print("Submission:",SUBMISSION)
else:
    print("No TEST_FILE configured; validation only.")


## Persist checkpoints before leaving Kaggle

Important artifacts live under:

```text
/kaggle/working/LLM-APA-runs/
```

After a meaningful run:

1. **Save Version**.
2. Enable saving Notebook Output.
3. In a later session, attach that saved output.
4. Set `PREVIOUS_RUN_DIR` to the prior run folder.
5. Use `RESUME_LATEST=True`.

Do not assume `/kaggle/working` survives after an ended session unless its output was saved/versioned.


In [ ]:
print("Run artifacts:")
for p in sorted(RUN_DIR.iterdir()):
    print(("DIR " if p.is_dir() else "FILE"), p.name)
print("\nSave Version with output enabled before ending the session.")
